
# SPENDDNA - Your Wallet's Year-End Story
# Week 2 Minor Project - The Unlox Academy


---


# This notebook analyzes Rahul Sharma's UPI/Bank transactions (Jan-Jun 2024)
# Built for Colab - Run all cells sequentially


---


# Done by - SHOYAL HALDAR
# Date : 3-july-2026

# SpendDNA: Spotify Wrapped for Your Money 💰
**Analyzing 6 months of Rahul Sharma's transactions**

*Built as part of The Unlox Academy Week 2 Minor Project*

**Features Implemented:**
- Transaction Parser
- Vendor Extractor
- Category Tagger
- Spending Overview
- Monthly Trends
- Time-of-Day Patterns
- Anomaly Detection
- Spending Archetypes

Imports

In [73]:
import pandas as pd
import numpy as np
from datetime import datetime

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


#Load & Parse Data (Feature 1 - Transaction Parser)

In [74]:
# Feature 1: Transaction Parser
# Feature 1: Transaction Parser
df = pd.read_csv('/content/Data set for DADS June.csv')
df = df.drop_duplicates().reset_index(drop=True)

# --- Date: 4 formats mixed in one column (ISO, DD/MM/YY, DD-Mon-YY, DD Mon YYYY) ---
# A single pd.to_datetime(dayfirst=True) call can't handle all 4 at once —
# it either fails to parse most rows, or worse, silently misreads the
# unambiguous ISO rows. So each format is detected by string shape first,
# then parsed with its own exact format.
s = df['Date'].astype(str).str.strip()
is_iso   = s.str.len().eq(10) & s.str[4].eq('-')   # 2024-01-01
is_slash = s.str.contains('/')                      # 02/01/24
is_dmy3  = s.str.contains('-') & ~is_iso             # 01-Jan-24
is_dmy4  = s.str.contains(' ')                       # 01 Jan 2024

parsed = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
parsed[is_iso]   = pd.to_datetime(s[is_iso],   format='%Y-%m-%d', errors='coerce')
parsed[is_slash] = pd.to_datetime(s[is_slash], format='%d/%m/%y', errors='coerce')
parsed[is_dmy3]  = pd.to_datetime(s[is_dmy3],  format='%d-%b-%y', errors='coerce')
parsed[is_dmy4]  = pd.to_datetime(s[is_dmy4],  format='%d %b %Y', errors='coerce')
df['Date'] = parsed
assert df['Date'].isnull().sum() == 0, "Unparsed dates remain"

# --- Amount: strip currency symbols/text and commas ---
amt = (df['Amount'].astype(str).str.strip()
       .str.replace('₹', '', regex=False)
       .str.replace('Rs.', '', regex=False)
       .str.replace(',', '', regex=False).str.strip())
df['Amount'] = pd.to_numeric(amt, errors='coerce')
assert df['Amount'].isnull().sum() == 0, "Unparsed amounts remain"

# --- Type: standardize to Debit/Credit ---
df['Type'] = df['Type'].map({'DR': 'Debit', 'Debit': 'Debit', 'CR': 'Credit'})
assert df['Type'].isnull().sum() == 0

print(f"✅ Parsed {len(df)} transactions")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")

✅ Parsed 1310 transactions
Date Range: 2024-01-01 to 2024-06-30


#Vendor Extractor (Feature 2)

In [75]:
#Vendor Extractor (Feature 2)

# Ai assested : understanding the approch and logic visulization.
def extract_vendor(desc):
    if not isinstance(desc, str): return 'UNCATEGORIZED'
    d = desc.upper()

    if any(x in d for x in ['INSTAMART', 'KIRANAKART']): return 'INSTAMART'

    if any(x in d for x in ['SWIGGY','BUNDL']): return 'SWIGGY'
    if 'ZOMATO' in d: return 'ZOMATO'
    if any(x in d for x in ['OLA','ANI TECHNOLOGIES']): return 'OLA'
    if any(x in d for x in ['AMAZON','AMZN']): return 'AMAZON'
    if any(x in d for x in ['RESTAURANT','EMPIRE','MEGHANA','TRUFFLES','DINEOUT']): return 'RESTAURANT'
    if 'ZEPTO' in d: return 'ZEPTO'
    if 'UBER' in d: return 'UBER'
    if any(x in d for x in ['BLINKIT','GROFERS']): return 'BLINKIT'
    if any(x in d for x in ['RAPIDO','ROPPEN']): return 'RAPIDO'
    if 'FLIPKART' in d or 'FKART' in d: return 'FLIPKART'
    if 'STARBUCKS' in d: return 'STARBUCKS'
    if any(x in d for x in ['BMTC','TUMMOC']): return 'BMTC'
    if any(x in d for x in ['THIRD WAVE','THIRDWAVE','TWC INDIA']): return 'THIRD_WAVE'
    if any(x in d for x in ['BPCL','HP PETROL','INDIAN OIL','IOC']): return 'FUEL'
    if any(x in d for x in ['NYKAA','FSN E-COMMERCE','INNOVATIVE RETAIL']): return 'NYKAA'
    if any(x in d for x in ['CAFE COFFEE DAY','COFFEE DAY GLOBAL','CCD']): return 'CCD'
    if any(x in d for x in ['DMART','AVENUE SUPERMARTS']): return 'DMART'
    if 'MYNTRA' in d: return 'MYNTRA'
    if any(x in d for x in ['BESCOM','BANGALORE ELEC','BWSSB','AIRTEL','JIO','VODAFONE','VI ']): return 'UTILITIES'
    if any(x in d for x in ['NETFLIX','HOTSTAR','STAR INDIA','SPOTIFY']): return 'SUBSCRIPTION'
    if 'BIGBASKET' in d: return 'BIGBASKET'
    if any(x in d for x in ['ZERODHA','GROWW','NEXTBILLION']): return 'INVESTMENT'
    if any(x in d for x in ['BOOKMYSHOW','BMS MOVIE','BIGTREE']): return 'BOOKMYSHOW'
    if 'RENT-LANDLORD' in d: return 'RENT'
    if 'SALARY' in d: return 'SALARY'
    if 'ATM' in d: return 'CASH'
    if any(('UPI-'+name) in d for name in ['AMAN','ANKIT','KARAN','NEHA','PRIYA','SNEHA','VIKAS']):
        return 'P2P'
    return 'OTHER'

# Classify each UNIQUE description once (283 of them), then map back onto
# all 1,310 rows -- avoids re-running the same if/elif chain on repeated
# descriptions like "BHIM SWIGGY" that appear dozens of times.
_unique_lookup = {d: extract_vendor(d) for d in df['Description'].unique()}
df['vendor_clean'] = df['Description'].map(_unique_lookup)

#Category Tagger (Feature 3)

In [76]:
# Feature 3: Category Tagger
# Covers every label vendor_clean can produce. 'SALARY' -> 'Income' (it's a
# credit, not spend -- must never sit inside a spend category, or it will
# inflate that category's total and distort savings rate). 'RENT' -> 'Housing'
# (no slot in the original 12 categories, given its own bucket instead of
# silently folding into 'Other').
cat_map = {
    'SWIGGY':'Food Delivery', 'ZOMATO':'Food Delivery', 'RESTAURANT':'Restaurants',
    'BLINKIT':'Quick Commerce', 'ZEPTO':'Quick Commerce',
    'BIGBASKET':'Groceries', 'DMART':'Groceries',
    'AMAZON':'E-commerce', 'FLIPKART':'E-commerce', 'MYNTRA':'E-commerce', 'NYKAA':'E-commerce',
    'UBER':'Transport', 'OLA':'Transport', 'RAPIDO':'Transport', 'BMTC':'Transport',
    'FUEL':'Fuel',
    'STARBUCKS':'Cafe', 'CCD':'Cafe', 'THIRD_WAVE':'Cafe',
    'SUBSCRIPTION':'Subscriptions',
    'UTILITIES':'Utilities',
    'BOOKMYSHOW':'Entertainment',
    'INVESTMENT':'Investments',
    'P2P':'Personal Transfer',
    'CASH':'Cash Withdrawal',
    'RENT':'Housing',
    'SALARY':'Income',
    'INSTAMART': 'Quick Commerce'
}
df['category'] = df['vendor_clean'].map(cat_map).fillna('Other')

print("Category Distribution:")
print(df['category'].value_counts())

Category Distribution:
category
Food Delivery        297
Transport            250
Quick Commerce       193
E-commerce           180
Cafe                  99
Restaurants           73
Utilities             40
Groceries             33
Subscriptions         31
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Housing                6
Other                  3
Name: count, dtype: int64


#Spending Overview (Feature 4)

In [77]:
# Feature 4: Spending Overview
debits = df[df['Type'] == 'Debit']    # capital D -- matches how F1 standardized Type
credits = df[df['Type'] == 'Credit']  # capital C

total_debits = debits['Amount'].sum()
total_credits = credits['Amount'].sum()
net_savings = total_credits - total_debits
savings_rate = (net_savings / total_credits * 100) if total_credits > 0 else 0

print("=== SPENDING OVERVIEW ===")
print(f"Total Credits: ₹{total_credits:,.0f}")
print(f"Total Debits: ₹{total_debits:,.0f}")
print(f"Net Savings: ₹{net_savings:,.0f}")
print(f"Savings Rate: {savings_rate:.1f}%")
print(f"Total Transactions: {len(df)}")
print("\nTop Categories by Spend:")
print(debits.groupby('category')['Amount'].sum().sort_values(ascending=False).head(8))

=== SPENDING OVERVIEW ===
Total Credits: ₹509,774
Total Debits: ₹1,678,901
Net Savings: ₹-1,169,127
Savings Rate: -229.3%
Total Transactions: 1310

Top Categories by Spend:
category
E-commerce        613076.0
Investments       248160.0
Food Delivery     129054.0
Restaurants       117737.0
Housing           108000.0
Quick Commerce     95667.0
Fuel               89303.0
Transport          57474.0
Name: Amount, dtype: float64


#Monthly Trend (Feature 5)

In [78]:
# Feature 5: Monthly Trend Analysis
df['Month'] = df['Date'].dt.month              # numeric, sorts correctly
debits = df[df['Type'] == 'Debit'].copy()
debits['Month'] = debits['Date'].dt.month

monthly_spend = debits.pivot_table(
    values='Amount',
    index='category',
    columns='Month',
    aggfunc='sum',
    fill_value=0
).rename(columns=lambda m: pd.Timestamp(2024, m, 1).strftime('%b'))  # relabel Jan..Jun, still in order

print("Monthly Spending Trend (Top Categories):")
top_cats = debits.groupby('category')['Amount'].sum().nlargest(5).index
print(monthly_spend.loc[top_cats])

Monthly Spending Trend (Top Categories):
Month              Jan      Feb       Mar      Apr      May       Jun
category                                                             
E-commerce     98623.0  95065.0  108981.0  74742.0  97632.0  138033.0
Investments    38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Food Delivery  20890.0  21452.0   20850.0  23054.0  22167.0   20641.0
Restaurants    16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Housing        18000.0  18000.0   18000.0  18000.0  18000.0   18000.0


# Time-of-Day Patterns (Feature 6)


In [79]:
# Feature 6: Time-of-Day Patterns
debits = df[df['Type'] == 'Debit'].copy()   # capital D

debits['Hour'] = pd.to_datetime(debits['Time'], format='%H:%M', errors='coerce').dt.hour

food_late = debits[(debits['category'] == 'Food Delivery') &
                   (debits['Hour'].notna()) &
                   ((debits['Hour'] >= 21) | (debits['Hour'] <= 2))]

denom = len(debits[debits['category'] == 'Food Delivery'])
late_pct = (len(food_late) / denom) * 100 if denom > 0 else 0

print(f"Late Night Food Delivery (9PM-2AM): {late_pct:.1f}%")

Late Night Food Delivery (9PM-2AM): 21.2%


#Anomaly Detection (Feature 7)

In [80]:
# ai assested : learing the z score type and implementiation

# Feature 7: Anomaly Detection
debits = df[df['Type'] == 'Debit'].copy()   # capital D

debits['category_mean'] = debits.groupby('category')['Amount'].transform('mean')
debits['category_std'] = debits.groupby('category')['Amount'].transform('std')
debits['z_score'] = (debits['Amount'] - debits['category_mean']) / debits['category_std']

anomalies = debits[debits['z_score'] > 2].sort_values('z_score', ascending=False)
print("Top Anomalies:")
print(anomalies[['Date', 'vendor_clean', 'category', 'Amount', 'z_score']].head(8))

Top Anomalies:
           Date vendor_clean     category   Amount   z_score
1280 2024-06-26       AMAZON   E-commerce  22008.0  4.182130
266  2024-02-07       AMAZON   E-commerce  21986.0  4.177184
409  2024-02-26   RESTAURANT  Restaurants   8383.0  3.884639
470  2024-03-05       AMAZON   E-commerce  19917.0  3.712029
1253 2024-06-22   RESTAURANT  Restaurants   7935.0  3.627582
646  2024-03-31   RESTAURANT  Restaurants   7931.0  3.625287
459  2024-03-04   RESTAURANT  Restaurants   7441.0  3.344131
445  2024-03-02       AMAZON   E-commerce  18273.0  3.342423


#Archetypes (Feature 8)

In [81]:
# ai assessted : desigin the architype what paramiter shuold i used and approch and formulation.

# Feature 8: Spending Archetype Detection

# 'hour' column pehle bana lo, kyunki abhi tak nahi hai
df['hour'] = df['Time'].str[:2].astype(int)

debits  = df[df['Type'] == 'Debit'].copy()
credits = df[df['Type'] == 'Credit'].copy()

# exclude non-consumption categories from % calculations
spend = debits[~debits['category'].isin(['Personal Transfer', 'Cash Withdrawal'])]
total_spend = spend['Amount'].sum()

def pct(cat_list):
    return spend[spend['category'].isin(cat_list)]['Amount'].sum() / total_spend * 100

food_pct      = pct(['Food Delivery', 'Restaurants', 'Cafe'])
qc_pct        = pct(['Quick Commerce'])
ecom_pct      = pct(['E-commerce'])
inv_pct       = pct(['Investments'])
transport_pct = pct(['Transport'])

total_credits = credits['Amount'].sum()
total_debits  = debits['Amount'].sum()
savings_rate  = (total_credits - total_debits) / total_credits * 100

def late_night_snacker_pct():
    fd = debits[debits['category'] == 'Food Delivery']
    if len(fd) == 0:
        return 0
    late = fd[(fd['hour'] >= 21) | (fd['hour'] <= 2)]
    return len(late) / len(fd) * 100

late_pct = late_night_snacker_pct()
sub_vendor_count = debits[debits['category'] == 'Subscriptions']['vendor_clean'].nunique()

def detect_foodie():             return food_pct > 25
def detect_quick_commerce():     return qc_pct > 15
def detect_shopaholic():         return ecom_pct > 15
def detect_investor():           return inv_pct > 15
def detect_late_night():         return late_pct > 50
def detect_cab_commuter():       return transport_pct > 10
def detect_subscription_lover(): return sub_vendor_count >= 5
def detect_yolo():               return savings_rate < 10
def detect_disciplined_saver():  return savings_rate > 40

archetypes = []
if detect_foodie():             archetypes.append(f"THE FOODIE ({food_pct:.1f}% on food)")
if detect_quick_commerce():     archetypes.append(f"THE QUICK COMMERCE JUNKIE ({qc_pct:.1f}%)")
if detect_shopaholic():         archetypes.append(f"THE SHOPAHOLIC ({ecom_pct:.1f}%)")
if detect_investor():           archetypes.append(f"THE INVESTOR ({inv_pct:.1f}%)")
if detect_late_night():         archetypes.append(f"THE LATE-NIGHT SNACKER ({late_pct:.1f}% food after 9PM)")
if detect_cab_commuter():       archetypes.append(f"THE CAB COMMUTER ({transport_pct:.1f}%)")
if detect_subscription_lover(): archetypes.append(f"THE SUBSCRIPTION LOVER ({sub_vendor_count} active subs)")
if detect_yolo():               archetypes.append(f"THE YOLO SPENDER (savings rate {savings_rate:.1f}%)")
if detect_disciplined_saver():  archetypes.append(f"THE DISCIPLINED SAVER (savings rate {savings_rate:.1f}%)")

print("=== SPENDING ARCHETYPES ===")
for a in archetypes:
    print(f"-> {a}")

print(f"\n(Supporting Stats — Food: {food_pct:.1f}%, Quick Commerce: {qc_pct:.1f}%, "
      f"E-commerce: {ecom_pct:.1f}%, Investments: {inv_pct:.1f}%, "
      f"Late-night: {late_pct:.1f}%, Savings rate: {savings_rate:.1f}%)")

=== SPENDING ARCHETYPES ===
-> THE SHOPAHOLIC (38.1%)
-> THE INVESTOR (15.4%)
-> THE YOLO SPENDER (savings rate -229.3%)

(Supporting Stats — Food: 17.3%, Quick Commerce: 5.9%, E-commerce: 38.1%, Investments: 15.4%, Late-night: 21.2%, Savings rate: -229.3%)



# FINAL SPENDDNA REPORT - Beautiful & Statistical Summary


In [83]:
# ai assested : help in formating

# Final Report

debits  = df[df['Type'] == 'Debit'].copy()
credits = df[df['Type'] == 'Credit'].copy()

total_credits = credits['Amount'].sum()
total_debits  = debits['Amount'].sum()
net_change    = total_credits - total_debits
savings_rate  = net_change / total_credits * 100

# consumption-only spend (excludes P2P transfers & cash withdrawals)
spend = debits[~debits['category'].isin(['Personal Transfer', 'Cash Withdrawal'])]
total_spend = spend['Amount'].sum()

print("=" * 70)
print(" SpendDNA REPORT — RAHUL SHARMA")
print(f" 6 months • {len(df):,} transactions • Jan to Jun 2024")
print("=" * 70)

# ---------------- EXECUTIVE SUMMARY ----------------
print("\n EXECUTIVE SUMMARY")
print(f" Total credits   : Rs. {total_credits:,.0f}")
print(f" Total debits    : Rs. {total_debits:,.0f}")
print(f" Net change      : Rs. {net_change:,.0f} ({'overspending' if net_change < 0 else 'saving'})")
tag = "🔥 BURNING SAVINGS" if savings_rate < 0 else ("👍 HEALTHY" if savings_rate > 20 else "⚠️ TIGHT")
print(f" Savings rate    : {savings_rate:.1f}% {tag}")
print(f" Transactions    : {len(df):,}")
print(f" Unique vendors  : {df['vendor_clean'].nunique()}")

# ---------------- TOP CATEGORIES ----------------
print("\n TOP CATEGORIES (% of consumption spend)")
cat_spend = spend.groupby('category')['Amount'].sum().sort_values(ascending=False)
for cat, amt in cat_spend.head(7).items():
    pct = amt / total_spend * 100
    bars = "#" * int(pct / 3)
    print(f" {cat:<16} {bars:<24} {pct:5.1f}%  Rs. {amt:>10,.0f}")

# ---------------- TOP VENDORS ----------------
print("\n TOP VENDORS")
vend = spend.groupby('vendor_clean').agg(total=('Amount', 'sum'), orders=('Amount', 'count'))
vend = vend.sort_values('total', ascending=False).head(5)
for name, row in vend.iterrows():
    print(f" {name:<15} Rs. {row['total']:>9,.0f}  ({int(row['orders']):>3} orders)")

# ---------------- TIME-OF-DAY PATTERNS ----------------
print("\n TIME-OF-DAY PATTERNS")
fd = debits[debits['category'] == 'Food Delivery']
late_pct = ((fd['hour'] >= 21) | (fd['hour'] <= 2)).mean() * 100 if len(fd) else 0
print(f" Food Delivery peaks   : 9 PM – 2 AM  ({late_pct:.1f}% of orders)")

cafe = debits[debits['category'] == 'Cafe']
morning_pct = ((cafe['hour'] >= 8) & (cafe['hour'] <= 11)).mean() * 100 if len(cafe) else 0
print(f" Cafe peaks            : 8 AM – 11 AM ({morning_pct:.1f}% of orders — morning coffee runs)")

qc = debits[debits['category'] == 'Quick Commerce']
qc_evening = ((qc['hour'] >= 18) & (qc['hour'] <= 22)).mean() * 100 if len(qc) else 0
print(f" Quick Commerce        : {qc_evening:.1f}% between 6–10 PM, rest spread through the day")

# ---------------- MONTHLY TREND (top category) ----------------
top_cat = cat_spend.index[0]
print(f"\n MONTHLY TREND ({top_cat})")
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun'}
monthly = spend[spend['category'] == top_cat].groupby('Month')['Amount'].sum()
max_val = monthly.max()
for m in range(1, 7):
    val = monthly.get(m, 0)
    bars = "#" * int(val / max_val * 20) if max_val else ""
    print(f" {month_names[m]}  Rs. {val:>9,.0f}  {bars}")

# ---------------- ANOMALIES ----------------
print("\n TOP ANOMALIES (z-score > 2, within category)")
debits['cat_mean'] = debits.groupby('category')['Amount'].transform('mean')
debits['cat_std']  = debits.groupby('category')['Amount'].transform('std')
debits['z_score']  = (debits['Amount'] - debits['cat_mean']) / debits['cat_std']
anomalies = debits[debits['z_score'] > 2].sort_values('z_score', ascending=False)
for _, r in anomalies.head(5).iterrows():
    print(f" {r['Date'].strftime('%d %b')} - {r['vendor_clean']:<12} Rs. {r['Amount']:>8,.0f}  (z={r['z_score']:.1f})")

# ---------------- ARCHETYPES ----------------
def pct_of(cat_list):
    return spend[spend['category'].isin(cat_list)]['Amount'].sum() / total_spend * 100

food_pct  = pct_of(['Food Delivery', 'Restaurants', 'Cafe'])
qc_pct    = pct_of(['Quick Commerce'])
ecom_pct  = pct_of(['E-commerce'])
inv_pct   = pct_of(['Investments'])
trans_pct = pct_of(['Transport'])
sub_count = debits[debits['category'] == 'Subscriptions']['vendor_clean'].nunique()

print("\n RAHUL'S SPENDING ARCHETYPES")
archetypes = []
if food_pct > 25:  archetypes.append(f"🍔 THE FOODIE ({food_pct:.1f}% on food)")
if qc_pct > 15:    archetypes.append(f"🚀 THE QUICK COMMERCE JUNKIE ({qc_pct:.1f}%)")
if ecom_pct > 15:  archetypes.append(f"🛍️ THE SHOPAHOLIC ({ecom_pct:.1f}%)")
if inv_pct > 15:   archetypes.append(f"📈 THE INVESTOR ({inv_pct:.1f}%)")
if late_pct > 50:  archetypes.append(f"🌙 THE LATE-NIGHT SNACKER ({late_pct:.1f}% food after 9PM)")
if trans_pct > 10: archetypes.append(f"🚕 THE CAB COMMUTER ({trans_pct:.1f}%)")
if sub_count >= 5: archetypes.append(f"📺 THE SUBSCRIPTION LOVER ({sub_count} active subs)")
if savings_rate < 10:  archetypes.append(f"🎉 THE YOLO SPENDER (savings rate {savings_rate:.1f}%)")
if savings_rate > 40:  archetypes.append(f"💰 THE DISCIPLINED SAVER (savings rate {savings_rate:.1f}%)")
for a in archetypes:
    print(f" -> {a}")

# ---------------- KEY INSIGHTS (dynamic, data-specific) ----------------
print("\n" + "=" * 70)
print(" KEY INSIGHTS")
monthly_burn = abs(net_change) / 6
print(f" 1. Rahul is burning through savings at Rs. {monthly_burn:,.0f}/month")
print(f"    -> Savings rate of {savings_rate:.1f}% is unsustainable long-term.")
print(f" 2. {top_cat} is the single biggest category at {cat_spend.iloc[0]/total_spend*100:.1f}% of spend,")
print(f"    driven by fewer but high-ticket transactions (avg Rs. "
      f"{spend[spend['category']==top_cat]['Amount'].mean():,.0f}/order).")
print(f" 3. {late_pct:.1f}% of food-delivery orders happen after 9 PM,")
print("    suggesting late-working or single-living habits.")
print(f" 4. Investments (SIP + Zerodha) sit at {inv_pct:.1f}% — a healthy floor,")
print("    but discretionary categories still dominate the wallet.")
print("=" * 70)

 SpendDNA REPORT — RAHUL SHARMA
 6 months • 1,310 transactions • Jan to Jun 2024

 EXECUTIVE SUMMARY
 Total credits   : Rs. 509,774
 Total debits    : Rs. 1,678,901
 Net change      : Rs. -1,169,127 (overspending)
 Savings rate    : -229.3% 🔥 BURNING SAVINGS
 Transactions    : 1,310
 Unique vendors  : 29

 TOP CATEGORIES (% of consumption spend)
 E-commerce       ############              38.1%  Rs.    613,076
 Investments      #####                     15.4%  Rs.    248,160
 Food Delivery    ##                         8.0%  Rs.    129,054
 Restaurants      ##                         7.3%  Rs.    117,737
 Housing          ##                         6.7%  Rs.    108,000
 Quick Commerce   #                          5.9%  Rs.     95,667
 Fuel             #                          5.6%  Rs.     89,303

 TOP VENDORS
 AMAZON          Rs.   328,530  ( 86 orders)
 INVESTMENT      Rs.   248,160  ( 23 orders)
 FLIPKART        Rs.   177,510  ( 47 orders)
 RESTAURANT      Rs.   117,737  ( 73 orde